# 🧩 Webinar 1 — Encoding de Variables Categóricas
**Sprint 11 | Dataset: Titanic**

---
1. Setup y exploración
2. One-Hot Encoding y la trampa de dummies
3. ¿Cuándo hacer OHE y cuándo NO?
4. LabelEncoder vs OrdinalEncoder
5. Cardinalidad alta — otras opciones

---

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = sns.load_dataset('titanic')[['survived','sex','age','embarked','class','deck','alone']].copy()
df.head()

,survived,sex,age,embarked,class,deck,alone
0,0,male,22.0,S,Third,NaN,False
1,1,female,38.0,C,First,C,False
2,1,female,26.0,S,Third,NaN,True
3,1,female,35.0,S,First,C,False
4,0,male,35.0,S,Third,NaN,True


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   survived  891 non-null    int64   
 1   sex       891 non-null    object  
 2   age       714 non-null    float64 
 3   embarked  889 non-null    object  
 4   class     891 non-null    category
 5   deck      203 non-null    category
 6   alone     891 non-null    bool    
dtypes: bool(1), category(2), float64(1), int64(1), object(2)
memory usage: 31.1+ KB


In [5]:
# Valores únicos por columna categórica
cat_cols = df.select_dtypes(include=['object','category']).columns
for col in cat_cols:
    print(f"{col:10} | {df[col].nunique():2} únicos | {df[col].unique()[:6]}")

sex        |  2 únicos | ['male' 'female']
embarked   |  3 únicos | ['S' 'C' 'Q' nan]
class      |  3 únicos | ['Third', 'First', 'Second']
Categories (3, object): ['First', 'Second', 'Third']
deck       |  7 únicos | [NaN, 'C', 'E', 'G', 'D', 'A']
Categories (7, object): ['A', 'B', 'C', 'D', 'E', 'F', 'G']


---
## 1. One-Hot Encoding (OHE)

Convierte cada categoría en una columna binaria.  
**Cuándo usarlo:** variables **nominales** (sin orden) con cardinalidad baja-media.

| pasajero | emb_C | emb_Q | emb_S |
|---|---|---|---|
| 1 | 1 | 0 | 0 |
| 2 | 0 | 0 | 1 |

In [6]:
# OHE con pandas
ohe = pd.get_dummies(df['embarked'], prefix='emb', dtype=int)
ohe.head(8)

,emb_C,emb_Q,emb_S
0,0,0,1
1,1,0,0
2,0,0,1
3,0,0,1
4,0,0,1
5,0,1,0
6,0,0,1
7,0,0,1


In [7]:
# OHE con sklearn — más útil en pipelines
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
resultado = enc.fit_transform(df[['embarked']].fillna('Unknown'))
pd.DataFrame(resultado, columns=enc.get_feature_names_out()).head(8)

,embarked_C,embarked_Q,embarked_S,embarked_Unknown
0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0
3,0.0,0.0,1.0,0.0
4,0.0,0.0,1.0,0.0
5,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0
7,0.0,0.0,1.0,0.0


---
## 2. La trampa de dummies

Con **n** categorías solo necesitas **n-1** columnas.  
Si todas son 0 → es la categoría eliminada. La columna extra genera **multicolinealidad** en modelos lineales.

> ⚠️ En árboles (DecisionTree, RandomForest) **no importa** — ellos manejan redundancia solos.

In [8]:
# ❌ Con trampa — 3 columnas para 3 categorías
con_trampa = pd.get_dummies(df['embarked'], prefix='emb', dtype=int)
print("Con trampa:  ", con_trampa.columns.tolist())

# ✅ Sin trampa
sin_trampa = pd.get_dummies(df['embarked'], prefix='emb', drop_first=True, dtype=int)
print("Sin trampa:  ", sin_trampa.columns.tolist())

Con trampa:   ['emb_C', 'emb_Q', 'emb_S']
Sin trampa:   ['emb_Q', 'emb_S']


In [9]:
# En sklearn: drop='first' o drop='if_binary'
enc_drop = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
data_emb = df[['embarked']].fillna('Unknown')
print("Sin drop:",  OneHotEncoder(sparse_output=False).fit_transform(data_emb).shape[1], "columnas")
print("Con drop:",  enc_drop.fit_transform(data_emb).shape[1], "columnas")

Sin drop: 4 columnas
Con drop: 3 columnas


---
## 3. ¿Cuándo hacer OHE y cuándo NO?

| Situación | ¿OHE? | Alternativa |
|---|---|---|
| Nominal, ≤ ~15 categorías | ✅ | — |
| Variable ordinal (tiene orden real) | ❌ | Ordinal encoding |
| Cardinalidad alta (> 20 cats) | ⚠️ | Frequency / Target encoding |
| Árbol / RandomForest | Opcional | LabelEncoder sirve también |
| Regresión lineal / logística / KNN | ✅ | OHE + drop_first |

In [10]:
# Ver impacto de OHE en número de columnas
for col in ['sex', 'embarked', 'deck']:
    n = df[col].nunique()
    print(f"{col:10} | {n:2} categorías → {n} columnas OHE (o {n-1} con drop_first)")

sex        |  2 categorías → 2 columnas OHE (o 1 con drop_first)
embarked   |  3 categorías → 3 columnas OHE (o 2 con drop_first)
deck       |  7 categorías → 7 columnas OHE (o 6 con drop_first)


---
## 4. LabelEncoder vs OrdinalEncoder

| | `LabelEncoder` | `OrdinalEncoder` |
|---|---|---|
| Entrada | 1 columna (1D) | DataFrame (2D, múltiples cols) |
| Uso típico | Codificar el **target `y`** | Codificar **features `X`** |
| Define orden | ❌ No | ✅ Sí (`categories=[...]`) |
| Compatible con Pipeline | ❌ | ✅ |

> ⚠️ `LabelEncoder` en features asume orden ficticio — usar solo para el target.

In [11]:
# LabelEncoder — para el TARGET
le = LabelEncoder()
df['sex_le'] = le.fit_transform(df['sex'])
print("Clases:", le.classes_)  # female=0, male=1
df[['sex','sex_le']].drop_duplicates()

Clases: ['female' 'male']


,sex,sex_le
0,male,1
1,female,0


In [12]:
# OrdinalEncoder — para FEATURES
oe = OrdinalEncoder()
df['sex_oe'] = oe.fit_transform(df[['sex']])
print("Categorías:", oe.categories_)
df[['sex','sex_le','sex_oe']].drop_duplicates()

Categorías: [array(['female', 'male'], dtype=object)]


,sex,sex_le,sex_oe
0,male,1,1.0
1,female,0,0.0


In [13]:
# ⚠️ El problema: OrdinalEncoder en variables NOMINALES asigna orden ficticio
oe2 = OrdinalEncoder()
df['embarked_oe'] = oe2.fit_transform(df[['embarked']].fillna('Unknown'))

# C=0, Q=1, S=2 → el modelo asume S > Q > C (¡sin sentido geográfico!)
df[['embarked','embarked_oe']].drop_duplicates().sort_values('embarked_oe')

,embarked,embarked_oe
1,C,0.0
5,Q,1.0
0,S,2.0
61,NaN,3.0


---
## 5. Codificación ordinal: `map` vs `OrdinalEncoder`

Cuando la variable **sí tiene orden real**, tenemos dos caminos:

| Método | Cuándo usarlo |
|---|---|
| `pandas .map()` | Scripts rápidos, EDA, fuera de Pipeline |
| `OrdinalEncoder(categories=[...])` | Dentro de Pipeline sklearn |

In [14]:
# Variable ordinal real: 'class' (First > Second > Third)

# ✅ Opción 1: pandas map — explícito
orden = {'Third': 1, 'Second': 2, 'First': 3}
df['class_map'] = df['class'].map(orden)

df[['class','class_map']].drop_duplicates().sort_values('class_map')

,class,class_map
1,First,3
9,Second,2
0,Third,1


In [15]:
# ✅ Opción 2: OrdinalEncoder con orden definido
oe_ord = OrdinalEncoder(categories=[['Third','Second','First']])
df['class_oe'] = oe_ord.fit_transform(df[['class']])

# OrdinalEncoder empieza en 0, map empezó en 1 — ambos respetan el orden
df[['class','class_map','class_oe']].drop_duplicates().sort_values('class_map')

,class,class_map,class_oe
1,First,3,2.0
9,Second,2,1.0
0,Third,1,0.0


---
## 6. `.fit_transform` vs `.transform`

**Regla clave para evitar data leakage:**

| | `fit_transform` | `transform` |
|---|---|---|
| Qué hace | Aprende las categorías + transforma | Solo transforma con lo aprendido |
| Usar en | **Train** | **Val / Test** |

In [16]:
df_clean = df[['survived','sex','embarked','class']].dropna().copy()
X = df_clean.drop('survived', axis=1)
y = df_clean['survived']
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

In [17]:
# ❌ MAL: fit_transform en train y en test por separado
enc_mal = OrdinalEncoder()
X_train_mal = enc_mal.fit_transform(X_train)
X_test_mal  = enc_mal.fit_transform(X_test)   # ← re-fittea: el orden puede cambiar

# ✅ BIEN: fit solo en train, transform en ambos
enc_bien = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_enc = enc_bien.fit_transform(X_train)  # aprende de train
X_test_enc  = enc_bien.transform(X_test)        # aplica lo aprendido

print("Train shape:", X_train_enc.shape)
print("Test shape: ", X_test_enc.shape)

Train shape: (711, 3)
Test shape:  (178, 3)


In [18]:
# ¿Qué pasa si test tiene una categoría que train no vio?
X_test_nuevo = X_test.copy()
X_test_nuevo.iloc[0, X_test_nuevo.columns.get_loc('embarked')] = 'Z'  # categoría nueva

res = enc_bien.transform(X_test_nuevo)
print("Valor para 'Z' (desconocido):", res[0, X_test_nuevo.columns.get_loc('embarked')])
# → -1 porque configuramos unknown_value=-1

Valor para 'Z' (desconocido): -1.0


---
## 7. Cardinalidad alta — ¿qué opciones hay?

Con variables de alta cardinalidad (ciudad, apellido, código postal), OHE explota el número de columnas.

| Técnica | Idea | Requiere target |
|---|---|---|
| **Frequency Encoding** | Reemplaza cat por su frecuencia | ❌ No |
| **Target Encoding** | Reemplaza cat por media del target | ✅ Sí |
| **Agrupar en 'Otro'** | Colapsa categorías raras | ❌ No |

In [19]:
# Frequency Encoding — vectorizado
freq_map = df['deck'].value_counts(normalize=True)
df['deck_freq'] = df['deck'].map(freq_map)

df[['deck','deck_freq']].drop_duplicates().sort_values('deck_freq', ascending=False)

,deck,deck_freq
10,G,0.019704
66,F,0.064039
6,E,0.157635
21,D,0.162562
1,C,0.290640
31,B,0.231527
23,A,0.073892
0,NaN,NaN


In [21]:
# Target Encoding — media del target por categoría
# ⚠️ Siempre calcular solo sobre TRAIN
df_te = df[['survived','deck']].dropna().copy()
train_te = df_te.sample(frac=0.8, random_state=42)

target_map = train_te.groupby('deck')['survived'].mean()
df_te['deck_target'] = df_te['deck'].map(target_map)

print(target_map.sort_values(ascending=False))

deck
B    0.777778
D    0.758621
E    0.708333
G    0.666667
C    0.574468
F    0.555556
A    0.500000
Name: survived, dtype: float64


C:\Users\57304\AppData\Local\Temp\ipykernel_23832\228616778.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_map = train_te.groupby('deck')['survived'].mean()


---
## 🏁 Resumen

| Variable | Tipo | Encoder |
|---|---|---|
| `sex` | Nominal binaria | `OHE(drop='if_binary')` |
| `embarked` | Nominal, 3 cats | `OHE(drop='first')` |
| `class` | Ordinal con orden real | `OrdinalEncoder(categories=[...])` o `map` |
| `deck` | Alta cardinalidad | Frequency encoding o agrupación |
| `pclass` | Ya es número | Sin encoding |

**Reglas de oro:**
- `.fit_transform` → solo en **train**
- `.transform` → en **test/val**
- Hay orden real → `OrdinalEncoder` con orden explícito
- No hay orden → `OHE`
- Cardinalidad alta → frequency encoding o agrupar antes de OHE